# RANDOM FOREST 

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder


## Carga del dataset

In [11]:
df = pd.read_csv('../data/raw/EmployeeAttrition.csv')
print(df.shape)
df.head()

(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


Al tener variables no numéricas debemos transformarlas, en este caso vamos a optar por usar One-Hot Encoding para transformar los strings en ints para que el modelo sea capaz de interpretar las variables correctamente

In [20]:
#Eliminar columnas innecesarias
try:
    cols_to_drop = ['Over18', 'StandardHours', 'EmployeeCount', 'EmployeeNumber', 'JobLevel']
    df = df.drop(columns=cols_to_drop)

except:
    print(df.head())
# Label Encoding para binarias (incluyendo Attrition)
le = LabelEncoder()
binarias = ['Attrition', 'OverTime', 'Gender']
for col in binarias:
    df[col] = le.fit_transform(df[col])

# One-Hot Encoding para las demás (Nominales)
df = pd.get_dummies(df, columns=['BusinessTravel', 'Department', 'EducationField', 
                                 'JobRole', 'MaritalStatus'], drop_first=True)

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EnvironmentSatisfaction  \
0                 1          2  Life Sciences                        2   
1                 8          1  Life Sciences                        3   
2                 2          2          Other                        4   
3                 3          4  Life Sciences                        4   
4                 2          1        Medical                        1   

   Gender  ...  PerformanceRating  RelationshipSatisfaction StockOptionLevel  \
0  Female  ...                

In [21]:


# 1. Separar variables predictoras (X) y objetivo (y)
X = df.drop('Attrition', axis=1)
y = df['Attrition']

# 2. Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Crear el modelo con ajuste de balanceo
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

# 4. Entrenar
rf.fit(X_train, y_train)

# 5. Evaluar
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      1.00      0.94       255
           1       1.00      0.10      0.19        39

    accuracy                           0.88       294
   macro avg       0.94      0.55      0.56       294
weighted avg       0.90      0.88      0.84       294



Estos resultados muestran el síntoma clásico de un modelo que sufre por el desbalance de clases.

El modelo es excelente detectando a los empleados que se quedan (0), pero es pésimo detectando a los que se van (1). De hecho, ahora mismo el modelo no sirve para el objetivo de RRHH.

Vamos a desglosarlo fila por fila de forma sencilla:

La trampa del Accuracy (0.88)
Tu precisión global es del 88%. Suena espectacular, pero es una falsa alarma.
Como tienes 255 empleados que se quedan y solo 39 que se van (un total de 294), si el modelo simplemente dijera "Nadie se va nunca", acertaría el 87% de las veces. Tu modelo está haciendo casi exactamente eso.

Clase 1 (Empleados que SE VAN) ❌ Aquí está el problema
Precision (1.00): Significa que el modelo es ultrasónico: cada vez que dice "Este empleado se va", acierta el 100% de las veces. No tiene falsos positivos.

Recall (0.10): Aquí está el desastre. Significa que de los 39 empleados que realmente se iban, el modelo solo fue capaz de detectar al 10% (es decir, unas 4 personas). Al 90% restante no los vio venir (falsos negativos).

F1-Score (0.19): Es la media entre Precision y Recall. Al estar tan cerca de 0, confirma que el modelo es muy deficiente prediciendo las bajas.

Clase 0 (Empleados que SE QUEDAN)
Precision (0.88) y Recall (1.00): El modelo clasificó a casi todo el mundo como "se queda". Por eso encuentra al 100% de los que se quedan, a costa de meter en esa bolsa a los que sí se iban.